# 🧠 تدريب نموذج MouthLocNet

دليل كامل لتدريب النموذج

**تم التطوير بمساعدة Perplexity AI**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

print('✅ المكتبات جاهزة')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 1️⃣ إعداد البيانات

In [ ]:
class MouthLocDataset(Dataset):
    """Dataset لأصوات الفم"""
    
    def __init__(self, num_samples=10000):
        self.num_samples = num_samples
        
        # محاكاة بيانات
        self.audio = np.random.randn(num_samples, 4, 7680).astype(np.float32)
        self.positions = np.random.uniform(-0.03, 0.03, (num_samples, 3)).astype(np.float32)
        self.positions[:, 2] = np.random.uniform(0.03, 0.07, num_samples).astype(np.float32)
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        return self.audio[idx], self.positions[idx]

# إنشاء dataset
train_dataset = MouthLocDataset(num_samples=8000)
val_dataset = MouthLocDataset(num_samples=2000)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f'✅ Train samples: {len(train_dataset)}')
print(f'✅ Val samples: {len(val_dataset)}')

## 2️⃣ إنشاء النموذج

In [ ]:
from mouthlocnet import MouthLocNet, ModelConfig

# إعدادات
config = ModelConfig()
model = MouthLocNet(config)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print(f'✅ Model created on {device}')
print(f'✅ Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 3️⃣ التدريب

In [ ]:
# Loss و optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=10, factor=0.5)

# حلقة التدريب
num_epochs = 50
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0.0
    
    for audio, positions in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
        audio = audio.to(device)
        positions = positions.to(device)
        
        optimizer.zero_grad()
        predicted = model(audio)
        loss = criterion(predicted, positions)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Validation
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for audio, positions in val_loader:
            audio = audio.to(device)
            positions = positions.to(device)
            
            predicted = model(audio)
            loss = criterion(predicted, positions)
            val_loss += loss.item()
    
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    
    # Scheduler
    scheduler.step(val_loss)
    
    print(f'Epoch {epoch+1}: Train Loss = {train_loss:.6f}, Val Loss = {val_loss:.6f}')

# حفظ النموذج
torch.save({
    'model_state_dict': model.state_dict(),
    'config': config,
    'train_losses': train_losses,
    'val_losses': val_losses,
}, 'mouthloc_net_v2.pt')

print('✅ Training complete!')
print('✅ Model saved to mouthloc_net_v2.pt')

## 4️⃣ تصور النتائج

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(train_losses, label='Train Loss', linewidth=2)
ax.plot(val_losses, label='Val Loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (MSE)')
ax.set_title('Training Progress')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_progress.png', dpi=150, bbox_inches='tight')
print('✅ Saved: training_progress.png')
plt.show()